<a href="https://colab.research.google.com/github/hieultph/llm-agent-research-course/blob/main/experiments/1B-folk-triage/1B_generate_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1B · Folk phrasing and diacritic loss → under-triage · notebook A (GPU: open models)

> **Tóm tắt (VI):** Notebook này cho các mô hình mở trả lời 260 câu hỏi (52 ca lấy từ hai bộ công khai × 5 cách kể: y khoa / dân gian × có dấu / không dấu, và bản dân gian kèm "tôi tính chữa dân gian …"), rồi lưu từng câu trả lời vào Drive (`MyDrive/llm-research/1B-folk-triage/gen/`). Nếu Colab ngắt giữa chừng, chạy lại từ đầu: những câu đã xong sẽ được bỏ qua.
>
> **Cách chạy:** Runtime → Change runtime type → **A100** (nhanh nhất, khoảng 1–1,5 giờ) hoặc **L4** (khoảng 2,5–3 giờ). Thêm secret `HF_TOKEN` (Colab → 🔑) để chạy Llama-3.1-8B; nếu không có, bước đó tự bỏ qua. Sau đó Runtime → **Run all**.
>
> Chạy xong thì mở notebook B (`1B_api_judge_analysis.ipynb`) để chấm và phân tích.

**Design.** Same case, same signs, only the wording changes: `clin`, `clin_nd` (diacritics stripped), `folk`, `folk_nd`, plus `folk_rem` (folk + "I plan to <folk remedy>, is that OK?"). 52 public vignettes (Levine et al. 2024; Ramaswamy et al. 2026) with their original labels: 14 emergency, 10 urgent (C/D), 28 soon/self-care controls; every case has a remedy. Neutral system prompt, greedy decoding, 512 new tokens.

In [ ]:
!pip -q install -U transformers accelerate

import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research/1B-folk-triage"
    REPO = "/content/llm-agent-research-course"
    if os.path.exists(REPO):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hieultph/llm-agent-research-course", REPO], check=True)
    CODE = f"{REPO}/experiments/1B-folk-triage"
    def secret(k):
        try: return userdata.get(k)
        except Exception: return None
else:
    ROOT = os.environ.get("FT_ROOT", "outputs")
    CODE = os.environ.get("FT_CODE", ".")
    secret = os.environ.get
sys.path.insert(0, CODE)
import importlib, ft; importlib.reload(ft)
os.makedirs(ROOT, exist_ok=True)
CASES = ft.load_cases(f"{CODE}/data/cases.jsonl")
SYSTEMS = ft.system_prompts(ft.load_lexicon(f"{CODE}/data/folk_lexicon.csv"))
BY_UID = {c["uid"]: c for c in CASES}
print(len(CASES), "prompts ->", ROOT)

In [ ]:
import torch
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), f"{VRAM:.0f} GB")
BIG = VRAM > 30          # A100 / G4: larger batches
HF_TOKEN = secret("HF_TOKEN")

# hub id: batch size on L4 (doubled on A100). Small to large so cheap models finish first.
MODELS = {
    "Qwen/Qwen2.5-1.5B-Instruct": 32,
    "Qwen/Qwen2.5-3B-Instruct": 32,
    "arcee-ai/Arcee-VyLinh": 32,            # Vietnamese-adapted, Qwen2.5-3B base
    "Qwen/Qwen2.5-7B-Instruct": 16,
    "SeaLLMs/SeaLLMs-v3-7B-Chat": 16,       # Southeast-Asian-adapted, Qwen2 base
    "Qwen/Qwen3-8B": 16,                    # thinking disabled
    "meta-llama/Llama-3.1-8B-Instruct": 16, # gated: needs HF_TOKEN
}
MITIGATION_MODELS = ["Qwen/Qwen2.5-7B-Instruct", "SeaLLMs/SeaLLMs-v3-7B-Chat", "meta-llama/Llama-3.1-8B-Instruct"]
SAMPLING_MODELS = ["Qwen/Qwen2.5-7B-Instruct", "SeaLLMs/SeaLLMs-v3-7B-Chat"]
MAX_NEW = 512

## Step 1 · Main run: every model, neutral prompt, greedy

In [ ]:
def gated_ok(m):
    return not m.startswith("meta-llama/") or bool(HF_TOKEN)

for m, bs in MODELS.items():
    if not gated_ok(m):
        print("skip (no HF_TOKEN):", m); continue
    ft.run_hf(m, CASES, SYSTEMS["base"], f"{ROOT}/gen/{ft.run_name(m)}.jsonl", system_key="base",
              batch_size=bs * (2 if BIG else 1), max_new_tokens=MAX_NEW, hf_token=HF_TOKEN)

In [ ]:
# ✅ checks: every prompt answered once, few empty or truncated answers; show one stroke case per model
import pandas as pd
rows = []
for m in MODELS:
    g = ft.read_jsonl(f"{ROOT}/gen/{ft.run_name(m)}.jsonl")
    if not g: continue
    uids = [x["uid"] for x in g]
    rows.append({"model": m.split("/")[-1], "n": len(g), "unique": len(set(uids)),
                 "empty": sum(not x["response"] for x in g), "truncated": sum(x["truncated"] for x in g),
                 "median_tokens": pd.Series([x["n_tokens"] for x in g]).median()})
chk = pd.DataFrame(rows); display(chk)
assert (chk.n == len(CASES)).all() and (chk.unique == len(CASES)).all(), "❌ missing or duplicated answers: re-run step 1"
print("✅ all models answered all", len(CASES), "prompts")
for m in MODELS:
    g = {x["uid"]: x for x in ft.read_jsonl(f"{ROOT}/gen/{ft.run_name(m)}.jsonl")}
    if g:
        print(f"\n── {m}  E01_folk_nd\n", g["E01_folk_nd"]["response"][:500])

## Step 2 · Mitigations (RQ4): generic safety line (m1) and folk-term glossary (m2)
Same prompts, different system prompt, on three 7–8B models. The glossary comes from `data/folk_lexicon.csv`.

In [ ]:
for k in ("m1", "m2"):
    print(f"── {k}: {SYSTEMS[k][:200]}…")
    for m in MITIGATION_MODELS:
        if not gated_ok(m): continue
        ft.run_hf(m, CASES, SYSTEMS[k], f"{ROOT}/gen/{ft.run_name(m, k)}.jsonl", system_key=k,
                  batch_size=MODELS[m] * (2 if BIG else 1), max_new_tokens=MAX_NEW, hf_token=HF_TOKEN)

## Step 3 · Stability check: 5 samples at temperature 0.7 on two models

In [ ]:
for m in SAMPLING_MODELS:
    ft.run_hf(m, CASES, SYSTEMS["base"], f"{ROOT}/gen/{ft.run_name(m, 'base', 'sample5')}.jsonl", system_key="base",
              batch_size=max(4, MODELS[m] // 4) * (2 if BIG else 1), max_new_tokens=MAX_NEW, n_samples=5, seed=0)
print("✅ done. Runtime → Disconnect and delete runtime, then open notebook B.")